# Spectrum interpolation — the conservative line-noise option

Where ZapLine removes a spatial subspace, spectrum interpolation replaces amplitude in a narrow band while **preserving phase**. It touches no spatial structure at all, which makes it the safer choice when the line is stationary and you cannot afford to lose rank.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sfreq, n_ch, dur = 500.0, 16, 60.0
n = int(sfreq * dur); t = np.arange(n) / sfreq

X = rng.standard_normal((n_ch, n)) * 1e-5
v = rng.standard_normal(n_ch); v /= np.linalg.norm(v)
for h, amp in [(1, 3.0e-5), (2, 1.0e-5), (3, 5.0e-6)]:
    X += amp * np.outer(v, np.sin(2 * np.pi * 50.0 * h * t))

info = mne.create_info(n_ch, sfreq, "eeg")
raw = mne.io.RawArray(X, info, verbose="ERROR")
print("50 Hz plus two harmonics, stationary throughout")

In [ ]:
import inspect
from mne_denoise.spectrum_interpolation import SpectrumInterpolation

print(inspect.signature(SpectrumInterpolation.__init__))

In [ ]:
si = SpectrumInterpolation(sfreq=sfreq, line_freq=50.0, n_harmonics=3)
clean_si = si.fit_transform(raw.copy())

from mne_denoise.zapline import ZapLine
zap = ZapLine(sfreq=sfreq, line_freq=50.0, n_select="auto")
clean_zap = zap.fit_transform(raw.copy())

kw = dict(method="welch", fmin=1.0, fmax=200.0, n_fft=8192, verbose="ERROR")
fig, ax = plt.subplots(figsize=(10, 5.5))
for name, obj, colour in [("original", raw, "#333333"),
                          ("spectrum interpolation", clean_si, "#CC79A7"),
                          ("zapline", clean_zap, "#E69F00")]:
    p = obj.compute_psd(**kw)
    ax.semilogy(p.freqs, p.get_data().mean(0), label=name, lw=2.0, color=colour)
ax.set_xlim(20, 200); ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("PSD (V²/Hz)")
ax.set_title("Fundamental and harmonics"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"rank kept: original {np.linalg.matrix_rank(raw.get_data())}, "
      f"SI {np.linalg.matrix_rank(clean_si.get_data())}, "
      f"ZapLine {np.linalg.matrix_rank(clean_zap.get_data())}")

> Spectrum interpolation leaves the spatial rank untouched. ZapLine removes components, so it costs rank — usually worth it when the line is non-stationary or spatially structured, and not worth it when it isn't.